# 面试问题：LLM 训练数据怎样做 PII 治理、删除与记忆泄露审计？

**一句话回答。** 在训练前对每个文档记录来源、许可、同意、PII 检测/处置与数据集版本；高风险记录应删除或经过批准的最小化处理。删除请求需要 tombstone、重建/遗忘计划和可审计的过滤证明；训练后还要用授权的泄露评测监控，而不是假设“公开数据就不会被记住”。

本 Notebook 以小型、受控数据实现必要的数据合同、验证器和状态机。它不访问真实网站、文件或模型，也不把断言结果宣传成生产质量、安全保证或法律合规结论。

**资料入口。** [Extracting Training Data from Large Language Models](https://arxiv.org/abs/2012.07805) 展示了训练数据提取风险；本例只使用虚构标识符说明数据治理，不执行模型攻击。


In [ ]:
question = "LLM 训练数据 PII 治理"  # 执行本行的状态、计算或校验逻辑。
assert "PII" in question  # 执行本行的状态、计算或校验逻辑。
assert 12 / 3 == 4  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 数据契约同时记录来源、许可和处理状态

仅保存清洗后的文本无法回答“它来自哪里、是否可训练、能否删除”。每个 document 应有不可变 source id、license/consent、采集时间、检测版本和决策原因；原始敏感文本必须在受控系统中最小化存储。


In [ ]:
documents = [{"id": "d1", "text": "产品说明公开发布", "license": "allowed", "consent": True, "pii": (), "decision": None}, {"id": "d2", "text": "联系邮箱 demo@example.test", "license": "allowed", "consent": True, "pii": ("email",), "decision": None}, {"id": "d3", "text": "内部资料", "license": "restricted", "consent": False, "pii": (), "decision": None}]  # 执行本行的状态、计算或校验逻辑。
assert len(documents) == 3  # 执行本行的状态、计算或校验逻辑。
assert documents[1]["pii"] == ("email",)  # 执行本行的状态、计算或校验逻辑。
assert documents[2]["license"] == "restricted"  # 执行本行的状态、计算或校验逻辑。

## 2. 检测器输出 span/type/版本，而不是只给一个布尔值

教学以虚构邮箱模式演示检测结果。真实 PII 检测需多模型/规则、人审、语言覆盖和误报处理；检测器版本变化会改变训练集组成，因此需与数据集 revision 一起记录。


In [ ]:
def detect_email(text):  # 执行本行的状态、计算或校验逻辑。
    token = "demo@example.test"  # 执行本行的状态、计算或校验逻辑。
    return ({"type": "email", "start": text.find(token), "end": text.find(token) + len(token), "detector": "v1"},) if token in text else ()  # 执行本行的状态、计算或校验逻辑。
spans = detect_email(documents[1]["text"])  # 执行本行的状态、计算或校验逻辑。
assert spans[0]["type"] == "email"  # 执行本行的状态、计算或校验逻辑。
assert spans[0]["start"] == 5  # 执行本行的状态、计算或校验逻辑。
assert detect_email(documents[0]["text"]) == ()  # 执行本行的状态、计算或校验逻辑。

## 3. 处置策略应保守、可解释且可复放

是否删除、隔离、最小化替换或允许，取决于许可、同意、敏感类型和业务目的。不能把“能正则替换”当作自动合规；高风险/不确定记录应进入人工审核，而非静默进入预训练。


In [ ]:
def decide(document):  # 执行本行的状态、计算或校验逻辑。
    if document["license"] != "allowed" or not document["consent"]:  # 执行本行的状态、计算或校验逻辑。
        return "drop"  # 执行本行的状态、计算或校验逻辑。
    if document["pii"]:  # 执行本行的状态、计算或校验逻辑。
        return "review_or_minimize"  # 执行本行的状态、计算或校验逻辑。
    return "allow"  # 执行本行的状态、计算或校验逻辑。
decisions = {document["id"]: decide(document) for document in documents}  # 执行本行的状态、计算或校验逻辑。
assert decisions == {"d1": "allow", "d2": "review_or_minimize", "d3": "drop"}  # 执行本行的状态、计算或校验逻辑。
assert decisions["d2"] == "review_or_minimize"  # 执行本行的状态、计算或校验逻辑。
assert decisions["d3"] == "drop"  # 执行本行的状态、计算或校验逻辑。

## 4. 最小化处理保留不可逆审计而非秘密原文

本例替换虚构 span，并记录处理版本与原文 id。生产需要加密访问、访问控制、保留期限和法律流程；hash 也可能是个人数据或可关联标识，不应误以为它天然匿名。


In [ ]:
def minimize(text, spans_value):  # 执行本行的状态、计算或校验逻辑。
    result = text  # 执行本行的状态、计算或校验逻辑。
    for span in sorted(spans_value, key=lambda item: item["start"], reverse=True):  # 执行本行的状态、计算或校验逻辑。
        result = result[:span["start"]] + "[REDACTED]" + result[span["end"]:]  # 执行本行的状态、计算或校验逻辑。
    return result  # 执行本行的状态、计算或校验逻辑。
redacted = minimize(documents[1]["text"], spans)  # 执行本行的状态、计算或校验逻辑。
assert redacted == "联系邮箱 [REDACTED]"  # 执行本行的状态、计算或校验逻辑。
assert "demo@example.test" not in redacted  # 执行本行的状态、计算或校验逻辑。
assert "[REDACTED]" in redacted  # 执行本行的状态、计算或校验逻辑。

## 5. 训练清单只允许通过 policy 的数据集版本

训练 job 读取的是 manifest，而不是任意目录。manifest 应包含文档 id、处理版本、shard、dataset revision 和过滤规则；drop/tombstone 文档不应通过重采样或缓存重新进入。


In [ ]:
manifest = [{"id": "d1", "text": documents[0]["text"], "dataset": "set-v1"}, {"id": "d2", "text": redacted, "dataset": "set-v1"}]  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in manifest] == ["d1", "d2"]  # 执行本行的状态、计算或校验逻辑。
assert all(item["dataset"] == "set-v1" for item in manifest)  # 执行本行的状态、计算或校验逻辑。
assert "d3" not in {item["id"] for item in manifest}  # 执行本行的状态、计算或校验逻辑。

## 6. 删除请求写入 tombstone 并触发下游重建计划

删除不是从一个索引删文件就结束：去重簇、训练 shard、缓存、embedding、checkpoint、adapter 和评测集都可能引用该数据。tombstone 明确请求范围和状态，系统据此生成重建/遗忘/限制服务的可审计任务。


In [ ]:
tombstones = {"d2": {"reason": "deletion_request", "status": "pending_rebuild", "revision": "del-v1"}}  # 执行本行的状态、计算或校验逻辑。
def eligible(item, tombstone_map):  # 执行本行的状态、计算或校验逻辑。
    return item["id"] not in tombstone_map  # 执行本行的状态、计算或校验逻辑。
filtered_manifest = [item for item in manifest if eligible(item, tombstones)]  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in filtered_manifest] == ["d1"]  # 执行本行的状态、计算或校验逻辑。
assert tombstones["d2"]["status"] == "pending_rebuild"  # 执行本行的状态、计算或校验逻辑。
assert not eligible(manifest[1], tombstones)  # 执行本行的状态、计算或校验逻辑。

## 7. 训练后泄露监控只用授权 canary/测试样本

不能在生产中主动诱导模型泄露真实个人信息。可以使用已批准的 synthetic canary、输出 DLP 检测、用户报告和 red-team 测试来发现风险；命中后要记录模型版本、上下文和处置，而非把可疑文本写进日志。


In [ ]:
canary = "CANARY-XY-42"  # 执行本行的状态、计算或校验逻辑。
def leaked(output, canary_value):  # 执行本行的状态、计算或校验逻辑。
    return canary_value in output  # 执行本行的状态、计算或校验逻辑。
assert leaked("模型输出 CANARY-XY-42", canary)  # 执行本行的状态、计算或校验逻辑。
assert not leaked("正常回答", canary)  # 执行本行的状态、计算或校验逻辑。
assert "@" not in canary  # 执行本行的状态、计算或校验逻辑。

## 8. 指标与审计覆盖误报、漏报和删除闭环

需要测 PII detector precision/recall、人工复核负担、许可覆盖、drop/minimize 比例、tombstone 生效延迟、训练 manifest 泄漏和后训练 canary 命中。任何单项高分都不能替代全链路数据治理。


In [ ]:
def rate(hits, total):  # 执行本行的状态、计算或校验逻辑。
    return hits / total if total else 0.0  # 执行本行的状态、计算或校验逻辑。
assert rate(2, 4) == 0.5  # 执行本行的状态、计算或校验逻辑。
assert rate(0, 0) == 0.0  # 执行本行的状态、计算或校验逻辑。
assert all(item["id"] != "d2" for item in filtered_manifest)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试回答应覆盖训练前来源/许可/PII 检测与处置、训练时版本化 manifest、删除 tombstone 的下游影响，以及训练后授权的泄露监控。重点是证据链和可回滚流程，而不是声称一次正则脱敏就永久消除了记忆风险。
